In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *
from rapidfuzz import process
from rapidfuzz import fuzz

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


In [4]:
file_path = os.path.join(output_folder, "Fern_manually_matched_v3_raw_db_station.csv")
match_3_pd = pd.read_csv(file_path)


filtered = match_3_pd[match_3_pd["db_var_match"].notna()]

# Update 'Has_inserted' directly in the original DataFrame using the subset's index
match_3_pd.loc[filtered.index, 'Has_inserted'] = 'Yes'

## Update this into the csv.files
# === Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_manually_matched_v3_raw_db_station.csv")
match_3_pd.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")


✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_manually_matched_v3_raw_db_station.csv


In [5]:

subset_no = match_3_pd[match_3_pd["Has_inserted"] == "No"].copy()
subset_no

# === Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_manually_matched_v4_raw_db_station.csv")
subset_no.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")

subset_batch5 = subset_no
subset_batch5

✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_manually_matched_v4_raw_db_station.csv


,station_name,native_id,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,matched_name,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,Has_inserted,score
0,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Air Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,61.538462
1,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 75 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,44.444444
2,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 50 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,44.444444
3,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 25 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,44.444444
4,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,61.538462
5,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Air Temp 2,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,53.333333
7,BarrenWx,Barren,Barren,54.510444,-126.614341,1051,WC_cal,m³/m³,2021-07-09 12:00:00,2024-10-03 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,21.052632
10,BlackhawkWx,Blackhawk,Blackhawk,55.078885,-120.352171,962,Snow depth raw,cm,2012-05-24 13:00:00,2025-01-07 10:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,66.666667
12,BlackhawkWx,Blackhawk,Blackhawk,55.078885,-120.352171,962,WC_cal,m?/m?,2012-05-24 13:00:00,2025-01-07 10:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,21.052632
15,BoulderWx,Boulder Creek,BoulderCr,55.108173,-127.374740,385,WC_cal,m³/m³,2010-06-07 13:00:00,2024-08-07 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,21.052632


In [6]:
# Group by variable and list unique stations for each variable
variable_station_summary = (
    subset_batch5.groupby('variable')['station_name']
    .unique()
    .reset_index()
)

# Optional: make it easier to read by joining station names
variable_station_summary['station_name'] = variable_station_summary['station_name'].apply(lambda x: ', '.join(x))

# Display the summary
variable_station_summary




,variable,station_name
0,Air Temp,Atlin School
1,Air Temp 2,Atlin School
2,DewPt 2,"BowronPit, Caribou Pass, CoalmineWx, Kluskus, ..."
3,Gd Temp 25 cm,Atlin School
4,Gd Temp 50 cm,Atlin School
5,Gd Temp 75 cm,Atlin School
6,RH 2,"BowronPit, Caribou Pass, CoalmineWx, Kluskus, ..."
7,Rain raw,PinkWx
8,Rain2,DunsterWx
9,Sfc Temp,Atlin School


In [7]:
# Add an empty notes column
variable_station_summary["notes"] = ""

# Define which variable names should get a note
vars_with_suffix = ["DewPt 2", "RH 2", "Temp 2"]

# Fill notes for those variables
mask = variable_station_summary["variable"].isin(vars_with_suffix)
variable_station_summary.loc[mask, "notes"] = "Has corresponding variable without '2' in raw data"


mask = variable_station_summary["variable"].isin(['Rain2'])
variable_station_summary.loc[mask, "notes"] = "Has 'Rain' in raw data"

mask = variable_station_summary["variable"].isin(['Rain raw'])
variable_station_summary.loc[mask, "notes"] = "Has 'Rain' in raw data"


mask = variable_station_summary["variable"].isin(['Snow depth raw'])
variable_station_summary.loc[mask, "notes"] = "Has 'Snow depth' in raw data"


variable_station_summary

# === Save the updated CSV ===
output_file = os.path.join(output_folder, "unmatched_variable_stations.csv")
variable_station_summary.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")

✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/unmatched_variable_stations.csv


In [8]:
variable_station_summary

,variable,station_name,notes
0,Air Temp,Atlin School,
1,Air Temp 2,Atlin School,
2,DewPt 2,"BowronPit, Caribou Pass, CoalmineWx, Kluskus, ...",Has corresponding variable without '2' in raw ...
3,Gd Temp 25 cm,Atlin School,
4,Gd Temp 50 cm,Atlin School,
5,Gd Temp 75 cm,Atlin School,
6,RH 2,"BowronPit, Caribou Pass, CoalmineWx, Kluskus, ...",Has corresponding variable without '2' in raw ...
7,Rain raw,PinkWx,Has 'Rain' in raw data
8,Rain2,DunsterWx,Has 'Rain' in raw data
9,Sfc Temp,Atlin School,


WC_cal         calibrated soil water content?, cubic meter of water per cubic meter of soil